# Metrics CERCA

In [1]:
from docx import Document
import pandas as pd
import gender_guesser.detector as gender
from df2gspread import gspread2df as g2d

from tqdm import tqdm
tqdm.pandas()

In [2]:
interest_centers = ['BETA']
file_path = '../data/external/5_Bibliometria_SIRIS_081025/'

center_name = 'BETA/'
file_name = 'LIST OF SCIENTIFIC PUBLICATIONS_BETA_Updated'

doc = Document(file_path + 'BM_' + center_name + file_name + '.docx')
pubs = [para.text for para in doc.paragraphs]
DOI_tmp = [pub.split('DOI:', 1)[1].strip() for pub in pubs  if 'DOI:' in pub]
DOI_BETA = [(DOI.split('https://doi.org/', 1)[1].strip() if 'https://doi.org/' in DOI else DOI) for DOI in DOI_tmp]
df_whole = pd.DataFrame(DOI_BETA, columns = ['DOI'])
df_whole['Center'] = 'BETA'

df = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA_BETA.csv')
df

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center
0,10.2993/0278-0771-41.1.53,Eric Marcel Temba,6.0,middle,False,1.377242e+08,MG,False,BETA
1,10.2993/0278-0771-41.1.53,Eric Marcel Temba,6.0,middle,False,1.337311e+08,FI,False,BETA
2,10.1111/1365-2656.13689,Pau Colom,2.0,middle,False,4.210117e+09,ES,False,BETA
3,10.1038/s41597-024-03611-7,Amanda H. Korstjens,129.0,middle,False,9.300472e+06,GB,False,BETA
4,10.1038/s41597-024-03611-7,Christopher J. Watson,253.0,middle,False,6.334173e+07,CA,False,BETA
...,...,...,...,...,...,...,...,...,...
2114,10.3390/d13090454,Andreu Ubach,15.0,middle,False,4.210111e+09,ES,True,BETA
2115,10.3390/membranes12090848,Mabel Mora,8.0,middle,False,1.153047e+08,ES,False,BETA
2116,10.1016/j.jnc.2022.126177,Diogo F. Ferreira,1.0,first,True,1.825342e+08,PT,False,BETA
2117,10.1016/j.jnc.2022.126177,Diogo F. Ferreira,1.0,first,True,4.512925e+07,GB,False,BETA


In [3]:
df_whole.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA    146
dtype: int64

In [4]:
df[df.CERCA == True].drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA    120
dtype: int64

## Publications Number

In [5]:
print('The total percentage of publications analyzed is:', df.DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.9315068493150684


In [6]:
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA    136
dtype: int64

## % led publications

In [7]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.821917808219178


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [8]:
df_cerca = df[df.CERCA == True]
df_led = df_cerca[(df_cerca.author_position == 'first') | (df_cerca.author_position == 'last') |(df_cerca.is_corresponding == True) ]
df_led.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA    0.522059
dtype: float64

## \% publications with women from the centre as authors

In [9]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.821917808219178


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [10]:
gend = gender.Detector()

df_cerca = df[df.CERCA == True].dropna(subset = 'display_name') # TO DELETE NON FOUND AUTHORS

df_cerca['first_name'] = df_cerca['display_name'].str.split(' ').str[0]
df_cerca['gender'] = df_cerca.first_name.progress_apply(lambda x: gend.get_gender(x))

df_cerca.drop_duplicates('first_name').sort_values('first_name', ascending = False).to_csv('gender_check_BETA.csv', index = False)
df_cerca

100%|██████████| 330/330 [00:00<00:00, 247739.45it/s]


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center,first_name,gender
9,10.1177/09636625221123420,Miquel Carandell Baruzzi,1.0,first,True,4.210111e+09,ES,True,BETA,Miquel,male
10,10.1177/09636625221123420,Miquel Carandell Baruzzi,1.0,first,True,1.230449e+08,ES,True,BETA,Miquel,male
16,10.1002/ecm.1561,Javier Sala‐Garcia,2.0,middle,False,4.210120e+09,ES,True,BETA,Javier,mostly_male
17,10.1002/ecm.1561,Javier Sala‐Garcia,2.0,middle,False,4.210130e+09,ES,True,BETA,Javier,mostly_male
56,10.1021/acsestwater.1c00192,Arben Merkoçi,13.0,last,True,4.210093e+09,ES,True,BETA,Arben,male
...,...,...,...,...,...,...,...,...,...,...,...
2084,10.1111/1365-2656.13689,Andreu Ubach,8.0,middle,False,4.210111e+09,ES,True,BETA,Andreu,male
2094,10.1016/j.anbehav.2020.12.013,Andreu Ubach,2.0,middle,False,4.210111e+09,ES,True,BETA,Andreu,male
2099,10.1007/s00442-022-05188-7,Andreu Ubach,1.0,first,True,4.210111e+09,ES,True,BETA,Andreu,male
2103,10.1007/s10841-023-00496-6,Andreu Ubach,1.0,first,False,4.210111e+09,ES,True,BETA,Andreu,male


**We manually revise the classifier**

In [11]:
gender_check = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', 'GenderBETA', col_names = True, row_names = False)
df_cerca = df_cerca.merge(gender_check[['first_name', 'gender_check']], on='first_name', how='left')
df_cerca['gender'] = df_cerca.apply(lambda row: row['gender_check'] if row['gender_check'] != '' else row['gender'], axis = 1)
df_cerca

Not all requested scopes were granted by the authorization server, missing scopes https://spreadsheets.google.com/feeds, https://docs.google.com/feeds.


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center,first_name,gender,gender_check
0,10.1177/09636625221123420,Miquel Carandell Baruzzi,1.0,first,True,4.210111e+09,ES,True,BETA,Miquel,male,
1,10.1177/09636625221123420,Miquel Carandell Baruzzi,1.0,first,True,1.230449e+08,ES,True,BETA,Miquel,male,
2,10.1002/ecm.1561,Javier Sala‐Garcia,2.0,middle,False,4.210120e+09,ES,True,BETA,Javier,male,male
3,10.1002/ecm.1561,Javier Sala‐Garcia,2.0,middle,False,4.210130e+09,ES,True,BETA,Javier,male,male
4,10.1021/acsestwater.1c00192,Arben Merkoçi,13.0,last,True,4.210093e+09,ES,True,BETA,Arben,male,
...,...,...,...,...,...,...,...,...,...,...,...,...
325,10.1111/1365-2656.13689,Andreu Ubach,8.0,middle,False,4.210111e+09,ES,True,BETA,Andreu,male,
326,10.1016/j.anbehav.2020.12.013,Andreu Ubach,2.0,middle,False,4.210111e+09,ES,True,BETA,Andreu,male,
327,10.1007/s00442-022-05188-7,Andreu Ubach,1.0,first,True,4.210111e+09,ES,True,BETA,Andreu,male,
328,10.1007/s10841-023-00496-6,Andreu Ubach,1.0,first,False,4.210111e+09,ES,True,BETA,Andreu,male,


In [12]:
df_fem = df_cerca[df_cerca.gender.isin(['female'])]

df_fem.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA    0.485294
dtype: float64

## \% publications led by women from the centre as authors

In [13]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.821917808219178


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [14]:
df_fem_led = df_fem[(df_fem.author_position == 'first') | (df_fem.author_position == 'last') |(df_fem.is_corresponding == True) ]

df_fem_led.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA    0.191176
dtype: float64

## \% publications in collaboration with other CERCA centres

In [15]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.821917808219178


In [16]:
df_tmp = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA_v2.csv')
df_tmp = df_tmp[df_tmp.Center != 'BETA'].reset_index(drop = True)

df_tmp_2 = pd.concat((df_tmp, df)).drop_duplicates().reset_index(drop = True)
df_tmp_2.to_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA_v3.csv')
df_tmp_2

/tmp/ipykernel_5752/2132732917.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tmp = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA_v2.csv')


,Unnamed: 0,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center
0,0.0,10.1038/s41591-023-02610-2,José R. Banegas,88.0,middle,False,4.210155e+09,ES,False,ResearchMar
1,1.0,10.1038/s41591-023-02610-2,Charles Agyemang,47.0,middle,False,8.870644e+08,NL,False,ResearchMar
2,2.0,10.1038/s41591-023-02610-2,Andrew Wong,750.0,middle,False,4.512925e+07,GB,False,ResearchMar
3,3.0,10.1038/s41591-023-02610-2,Farhad Zamani,766.0,middle,False,1.611069e+08,IR,False,ResearchMar
4,4.0,10.1038/s41591-023-02610-2,José R. Banegas,88.0,middle,False,6.363444e+07,ES,False,ResearchMar
...,...,...,...,...,...,...,...,...,...,...
260417,NaN,10.3390/d13090454,Andreu Ubach,15.0,middle,False,4.210111e+09,ES,True,BETA
260418,NaN,10.3390/membranes12090848,Mabel Mora,8.0,middle,False,1.153047e+08,ES,False,BETA
260419,NaN,10.1016/j.jnc.2022.126177,Diogo F. Ferreira,1.0,first,True,1.825342e+08,PT,False,BETA
260420,NaN,10.1016/j.jnc.2022.126177,Diogo F. Ferreira,1.0,first,True,4.512925e+07,GB,False,BETA


In [17]:
institution = 'BETA'

In [20]:
cerca_centers = {'BETA' : ['115304700'], # NOT IN OA - WE'LL USE THE PREVIOUS RESULT; I HAVE CONSIDERED THE ONE WITH MOST PUBLICATIONS
                    'CREAF' : ['4210129656'],
                    'ICN2' : ['4210093216'],
                    'ISGlobal' : ['4210148332'],
                    'ResearchMar' : ['4210156109']}

df_cerca_af = pd.read_csv('../data/external/ToCheck - AffID.csv')

df_center = df[(df.Center == institution) & (df.institution_id != int(cerca_centers[institution][0]))]
df_colab = df_center[df_center.institution_id.isin(df_cerca_af.OA_id)] 
percentage = df_colab.DOI.nunique() / df_center.DOI.nunique()
print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

The percentage of publications in collaboration for BETA is: 11.76%


In [19]:
# df_unique = df_tmp_2[['DOI', 'Center', 'CERCA']].drop_duplicates()
# df_cerca = df_unique[df_unique['CERCA'] == True]
# dois_this = set(df_cerca.loc[df_cerca.Center == institution, 'DOI'])
# dois_others = set(df_cerca.loc[df_cerca.Center != institution, 'DOI'])
# collaborative_dois = dois_this & dois_others   
# percentage = len(collaborative_dois) / len(dois_this) if dois_this else 0
# print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

## \% publications in collaboration with other local institutions

In [22]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.821917808219178


In [24]:
cerca_centers = {'BETA' : ['115304700'], # NOT IN OA - WE'LL USE THE PREVIOUS RESULT; I HAVE CONSIDERED THE ONE WITH MOST PUBLICATIONS
                    'CREAF' : ['4210129656'],
                    'ICN2' : ['4210093216'],
                    'ISGlobal' : ['4210148332'],
                    'ResearchMar' : ['4210156109']}

df_cerca_af = pd.read_csv('../data/external/ToCheck - AffID.csv')

df_center = df[(df.Center == institution) & (df.institution_id != int(cerca_centers[institution][0]))]
df_cerca_colab = df_center[df_center.institution_id.isin(df_cerca_af.OA_id)]
df_not_cerca_colab = df_center[(~df_center.DOI.isin(df_cerca_colab.DOI)) & (df_center.COUNTRY_CODE == 'ES')]
percentage = df_not_cerca_colab.DOI.nunique() / df_center.DOI.nunique()
print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

The percentage of publications in collaboration for BETA is: 77.94%


In [ ]:
# df_unique = df_tmp_2[['DOI', 'Center', 'CERCA', 'COUNTRY_CODE']].drop_duplicates()
# df_cerca = df_unique[df_unique['CERCA'] == True]
# df_spanish_non_cerca = df_unique[(df_unique['CERCA'] == False) & (df_unique['COUNTRY_CODE'] == 'ES')]

# dois_center = set(df_cerca.loc[df_cerca['Center'] == institution, 'DOI'])
# dois_spanish_non_cerca = set(df_spanish_non_cerca['DOI'])
# collaborative_dois = dois_center & dois_spanish_non_cerca
# percentage = len(collaborative_dois) / len(dois_center)
# print(f'The percentage of publications analyzed for {institution} is: {percentage:.2%}')

The percentage of publications analyzed for BETA is: 71.67%


## \% publications in collaboration with other international institutions

In [31]:
df_unique = df[['DOI', 'Center', 'CERCA', 'COUNTRY_CODE']].drop_duplicates()

df_cerca = df_unique[df_unique['CERCA'] == True]
df_international_non_cerca = df_unique[(df_unique['CERCA'] == False) & (df_unique['COUNTRY_CODE'] != 'ES')]

dois_center = set(df_cerca.loc[df_cerca['Center'] == institution, 'DOI'])
dois_international = set(df_international_non_cerca['DOI'])

collaborative_dois = dois_center & dois_international
    
percentage = len(collaborative_dois) / len(dois_center) if dois_center else 0
print(f'The percentage of international collaborations for {institution} is: {percentage:.2%}')

The percentage of international collaborations for BETA is: 62.50%
